# Kaggle Training Notebook

This notebook runs the full backbone-training workflow on Kaggle, saves reusable artifacts, and finishes with metric, loss, and SR-vs-LR comparisons.


## Strategy

1. Prepare the Kaggle environment and install the repo dependencies.
2. Run the classical backbone feature-extraction sweep for SR and LR.
3. Run the deep regressor sweep so training histories are saved.
4. Load saved metrics and histories.
5. Plot train/validation metrics, deep loss curves, and the final SR-vs-LR comparison.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display

REPO = Path("/kaggle/working/S2-super-resolution")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/AhmedTrb/S2-super-resolution.git", str(REPO)], check=True)

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "torchgeo"], check=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
sns.set_theme(style="whitegrid", context="talk")

print("Working directory:", REPO)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

## Kaggle Setup

Clone the repository if needed, install dependencies, and confirm that CUDA is available before launching the experiment sweeps.


In [ ]:
MODEL_SPECS = [
    ("torchgeo:resnet50", "ResNet50_Weights.SENTINEL2_ALL_DINO", "resnet50_all_dino"),
    ("torchgeo:resnet50", "ResNet50_Weights.SENTINEL2_ALL_MOCO", "resnet50_all_moco"),
    ("torchgeo:resnet50", "ResNet50_Weights.SENTINEL2_ALL_DECUR", "resnet50_all_decur"),
    ("torchgeo:vit_small_patch16_224", "ViTSmall16_Weights.SENTINEL2_ALL_DINO", "vitsmall16_all_dino"),
    ("torchgeo:vit_small_patch16_224", "ViTSmall16_Weights.SENTINEL2_ALL_MOCO", "vitsmall16_all_moco"),
    ("torchgeo:vit_base_patch16_224", "ViTBase16_Weights.SENTINEL2_ALL_MAE", "vitbase16_all_mae"),
    ("torchgeo:vit_small_patch14_dinov2", "ViTSmall14_DINOv2_Weights.SENTINEL2_ALL_SOFTCON", "vitsmall14_softcon"),
    ("torchgeo:vit_base_patch14_dinov2", "ViTBase14_DINOv2_Weights.SENTINEL2_ALL_SOFTCON", "vitbase14_softcon"),
    ("torchgeo:resnet50", "ResNet50_Weights.SENTINEL2_MI_MS_SATLAS", "resnet50_mi_ms_satlas"),
    ("torchgeo:swin_v2_t", "Swin_V2_T_Weights.SENTINEL2_MI_MS_SATLAS", "swinv2t_mi_ms_satlas"),
]


def make_experiments(resolution):
    return [
        {
            "resolution": resolution,
            "backbone": backbone,
            "weight": weight,
            "run_name": f"{resolution}_{name}",
        }
        for backbone, weight, name in MODEL_SPECS
    ]


SR_EXPERIMENTS = make_experiments("sr")
LR_EXPERIMENTS = make_experiments("lr")
CLASSIC_REGRESSORS = ["ridge", "elastic_net", "pls", "random_forest", "extra_trees", "xgboost"]


REPO = Path("/kaggle/working/S2-super-resolution")
INVENTORY = REPO / "yellowness_dataset" / "observations_inventory.csv"
DATASET_DIR = REPO / "yellowness_dataset"
ML_ROOT = REPO / "outputs" / "yellowness_backbone_ml"
DEEP_ROOT = REPO / "outputs" / "yellowness_regression"
PLOT_ROOT = REPO / "outputs" / "kaggle_plots"
for path in (ML_ROOT, DEEP_ROOT, PLOT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

python_bin = sys.executable
COMMON_ML_ARGS = {
    "mask_fusion": "feature_mask_pool",
    "dr_methods": ["none", "pca"],
    "pca_variance": "0.95",
    "pls_top_k": "8",
    "batch_size": "32",
    "num_workers": "4",
}

COMMON_DEEP_ARGS = {
    "mask_fusion": "feature_mask_pool",
    "epochs": "40",
    "batch_size": "8",
    "num_workers": "4",
    "learning_rate": "1e-3",
    "backbone_learning_rate": "1e-4",
    "sample_patch_size": "224",
    "center_crop_size": "224",
}


def run_cmd(cmd, allow_failure=True):
    print("RUN:", " ".join(map(str, cmd)))
    try:
        subprocess.run([str(part) for part in cmd], check=True, cwd=REPO)
        return True
    except subprocess.CalledProcessError as exc:
        if not allow_failure:
            raise
        print(f"Skipping failed run: {exc}")
        return False


print(f"Repo: {REPO}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"Inventory: {INVENTORY}")

## Classical Backbone ML Sweep

This section runs the feature-extraction plus classical regression sweep for the full SR and LR experiment matrix.


In [ ]:
# Run the classical backbone-ML sweep end-to-end.
# Set RUN_FULL_SWEEP = False if you only want to inspect the configuration first.
RUN_FULL_SWEEP = True


def run_backbone_ml_experiments(experiments):
    for spec in experiments:
        out_dir = ML_ROOT / spec["run_name"]
        cmd = [
            python_bin,
            "scripts/train_yellowness_backbone_ml.py",
            "--inventory", str(INVENTORY),
            "--root-dir", str(DATASET_DIR),
            "--resolution", spec["resolution"],
            "--backbone", spec["backbone"],
            "--torchgeo-weight", spec["weight"],
            "--mask-fusion", "feature_mask_pool",
            "--regressors", *CLASSIC_REGRESSORS,
            "--dr-methods", "none", "pca",
            "--pca-variance", "0.95",
            "--pls-top-k", "8",
            "--batch-size", "32",
            "--num-workers", "4",
            "--save-feature-csv",
            "--feature-csv-name", f"{spec['run_name']}_embeddings.csv",
            "--export-deep-runs-summary",
            "--deep-runs-root", str(DEEP_ROOT),
            "--output-dir", str(out_dir),
        ]
        run_cmd(cmd)


if RUN_FULL_SWEEP:
    run_backbone_ml_experiments(SR_EXPERIMENTS)
    run_backbone_ml_experiments(LR_EXPERIMENTS)


## Deep Model Training

Run the end-to-end regressor training sweep for the same backbone families. These runs save checkpoints, run configs, and epoch histories that feed the loss-evolution plots at the end.


In [ ]:
# Run the deep-model sweep. This is slower than the classical feature pipeline, but it saves the training history needed for loss plots.

def run_deep_experiments(experiments, epochs=40, batch_size=8, num_workers=4):
    for spec in experiments:
        out_dir = DEEP_ROOT / spec["run_name"]
        cmd = [
            python_bin,
            "scripts/train_yellowness_regressor.py",
            "--inventory", str(INVENTORY),
            "--root-dir", str(DATASET_DIR),
            "--resolution", spec["resolution"],
            "--backbone", spec["backbone"],
            "--torchgeo-weight", spec["weight"],
            "--mask-fusion", "feature_mask_pool",
            "--freeze-backbone",
            "--unfreeze-epoch", "25",
            "--epochs", str(epochs),
            "--batch-size", str(batch_size),
            "--num-workers", str(num_workers),
            "--learning-rate", "1e-3",
            "--backbone-learning-rate", "1e-4",
            "--sample-patch-size", "224",
            "--center-crop-size", "224",
            "--rotate-augment",
            "--data-parallel",
            "--output-dir", str(out_dir),
        ]
        run_cmd(cmd)

# Uncomment one line at a time if you want to split the run into manageable chunks.
# run_deep_experiments(SR_EXPERIMENTS)
# run_deep_experiments(LR_EXPERIMENTS)


## Load Saved Results

Reload the saved classical-model summaries and deep-training histories before generating the comparison plots.


In [ ]:
def save_fig(fig, filename):
    path = PLOT_ROOT / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    return path


def load_ml_results():
    frames = []
    for path in sorted(ML_ROOT.glob("sr_*/summary.csv")) + sorted(ML_ROOT.glob("lr_*/summary.csv")):
        if not path.exists():
            continue
        df = pd.read_csv(path)
        df["run"] = path.parent.name
        df["resolution"] = "sr" if path.parent.name.startswith("sr_") else "lr"
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def load_deep_history():
    rows = []
    for path in sorted(DEEP_ROOT.glob("**/training_history.json")):
        run_dir = path.parent
        try:
            history = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        meta = {"run": run_dir.name}
        config_path = run_dir / "run_config.json"
        if config_path.exists():
            try:
                config = json.loads(config_path.read_text(encoding="utf-8"))
                meta.update(
                    {
                        "resolution": config.get("resolution"),
                        "backbone": config.get("backbone"),
                        "torchgeo_weight": config.get("torchgeo_weight"),
                        "mask_fusion": config.get("mask_fusion"),
                    }
                )
            except Exception:
                pass
        for row in history:
            rows.append({**meta, **row})
    return pd.DataFrame(rows)


ml_results = load_ml_results()
deep_history = load_deep_history()

display(ml_results.head())
display(deep_history.head())

In [ ]:
def best_ml_rows(df):
    if df.empty:
        return df
    key_cols = ["resolution", "backbone", "regressor"]
    best_idx = df.groupby(key_cols)["val_rmse"].idxmin()
    return df.loc[best_idx].reset_index(drop=True)


def plot_sr_vs_lr_metrics(df):
    if df.empty:
        print("No ML results found yet.")
        return pd.DataFrame(), pd.DataFrame()

    best = best_ml_rows(df)
    agg = best.groupby(["resolution", "regressor"], as_index=False)[["train_rmse", "val_rmse", "train_mae", "val_mae", "train_r2", "val_r2"]].mean()

    figure_specs = [
        ("train_rmse", "val_rmse", "RMSE"),
        ("train_mae", "val_mae", "MAE"),
        ("train_r2", "val_r2", "R2"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
    for ax, (train_col, val_col, metric_name) in zip(axes, figure_specs):
        melted = agg.melt(
            id_vars=["resolution", "regressor"],
            value_vars=[train_col, val_col],
            var_name="split",
            value_name="value",
        )
        melted["split"] = melted["split"].map({train_col: "train", val_col: "validation"})
        melted["series"] = melted["resolution"].str.upper() + " / " + melted["split"]
        sns.barplot(data=melted, x="regressor", y="value", hue="series", ax=ax, errorbar=None)
        ax.set_title(f"{metric_name}: train vs validation")
        ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    saved = save_fig(fig, "sr_vs_lr_metrics.png")
    print(f"Saved: {saved}")
    plt.show()

    return best, agg


def plot_backbone_comparison(df):
    if df.empty:
        return
    best = best_ml_rows(df)
    agg = best.groupby(["resolution", "backbone"], as_index=False)["val_rmse"].mean()
    fig, ax = plt.subplots(figsize=(18, 6))
    sns.barplot(data=agg, x="backbone", y="val_rmse", hue="resolution", ax=ax, errorbar=None)
    ax.set_title("SR vs LR by backbone (best validation RMSE per run)")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    saved = save_fig(fig, "sr_vs_lr_backbones.png")
    print(f"Saved: {saved}")
    plt.show()


def plot_deep_loss_evolution(history_df, top_n=6):
    if history_df.empty:
        print("No deep training history found yet.")
        return

    hist = history_df.copy()
    hist["epoch"] = hist["epoch"].astype(int)
    ranked_runs = (
        hist.groupby(["run", "resolution", "backbone"], as_index=False)["val_loss"]
        .min()
        .sort_values("val_loss")
        .head(top_n)
    )
    selected = hist.merge(ranked_runs[["run"]], on="run", how="inner")

    fig, axes = plt.subplots(len(ranked_runs), 1, figsize=(14, max(4, 4 * len(ranked_runs))), sharex=True)
    if len(ranked_runs) == 1:
        axes = [axes]

    for ax, run_name in zip(axes, ranked_runs["run"].tolist()):
        run_df = selected[selected["run"] == run_name].sort_values("epoch")
        sns.lineplot(data=run_df, x="epoch", y="train_loss", ax=ax, label="train")
        sns.lineplot(data=run_df, x="epoch", y="val_loss", ax=ax, label="validation")
        meta = run_df.iloc[0]
        ax.set_title(f"{meta.get('resolution', '')} | {meta.get('backbone', '')} | {run_name}")
        ax.set_ylabel("Loss")

    plt.tight_layout()
    saved = save_fig(fig, "deep_loss_evolution.png")
    print(f"Saved: {saved}")
    plt.show()


best_ml_results, ml_aggregated = plot_sr_vs_lr_metrics(ml_results)
plot_backbone_comparison(ml_results)
plot_deep_loss_evolution(deep_history)

## Outputs And Saved Artifacts

Each run writes its own model artifacts, metric tables, and figure files.

Saved outputs include:
- `summary.csv` and `summary.json` for per-run metrics
- `model.pkl` for the fitted classical model and preprocessing stack
- `best_model.pt`, `training_history.json`, and `run_config.json` for deep runs
- `backbone_embeddings.csv` and `backbone_embeddings.npz` for reusable embeddings
- saved plots under `outputs/kaggle_plots/`
